## LLM Zoomcamp
### Homework 1

Homework questions are at https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md

In [1]:
import sys
print(sys.version)

3.14.3 | packaged by conda-forge | (main, Feb  9 2026, 22:17:37) [Clang 20.1.8 ]


In [2]:
#!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
#!wget https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from anthropic import Anthropic
from gitsource import GithubRepositoryDataReader
from minsearch import Index

In [4]:
parent_dir = Path.cwd().parent
dotenv_path = parent_dir / '.env'
load_dotenv(dotenv_path = dotenv_path)
client = Anthropic(api_key = os.getenv("ANTHROPIC_API_KEY"))


## Q1. How many lesson pages


In [5]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [6]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [7]:
len(documents)

72

In [8]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q2. Indexing and searching

In [9]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [10]:
question='How does the agentic loop keep calling the model until it stops?'

In [11]:
index.search(question)[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

## Q3. RAG

In [12]:
from rag_helper import RAGBase

In [13]:
# Create a RAG instance:
rag = RAGBase(
    index=index,
    llm_client=client
)

In [14]:
# Call the rag method:
answer, input_tokens = rag.rag(question)
print(answer)



The agentic loop keeps calling the model until it stops by using a `while True` loop with a flag called `has_function_calls`. Here's how it works:

1. **Each iteration**, the loop sends the current message history to the model and gets a response.
2. It then inspects every item in the response output:
   - If an item is a **function call**, it executes the tool, appends the result back to the messages,


In [15]:
print(input_tokens)

8203


In [16]:
#help(RAGBase.__init__)

In [17]:
#import inspect
#from rag_helper import RAGBase
#print(inspect.getsource(RAGBase.__init__))

In [18]:
#print(rag.__dict__)

## Q4. Chunking

In [19]:
from gitsource import chunk_documents

In [20]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [21]:
chunks[1]

{'start': 1000,
 'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they sometim

In [22]:
len(chunks)

295

## Q5. RAG with chunking

In [23]:
chunks_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunks_index.fit(chunks)

In [24]:
# Create a RAG instance:
rag = RAGBase(
    index=chunks_index,
    llm_client=client
)

In [25]:
# Call the rag method:
answer, input_tokens = rag.rag(question)
print(answer)


## How the Agentic Loop Keeps Calling the Model Until It Stops

The agentic loop uses a `while True` loop with a simple exit condition based on whether the model made any function calls during the current iteration.

Here's how it works step by step:

1. **The loop starts** and sets a flag `has_function_calls = False` at the beginning of each iteration.

2. **The model is called** with the current


In [26]:
print(input_tokens)

2659


## Q6. Turning it into an agent

Sources:

https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/15-frameworks.md

https://github.com/alexeygrigorev/toyaikit/tree/main

In [27]:
from toyaikit.llm import AnthropicClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import AnthropicMessagesRunner, DisplayingRunnerCallback

In [28]:
def search(query: str) -> dict[str, str]:
    """
    Search the database for entries matching the given query.
    """
    return chunks_index.search(
        query,
        num_results=5,
        boost_dict={'content': 1.0, 'filename': 0.5}
    )


In [29]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [30]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [31]:
instructions = '''
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
'''

In [32]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = AnthropicMessagesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=AnthropicClient(model='claude-opus-4-6')
)

In [33]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback
)

-> Response received


-> Response received


-> Response received


-> Response received


,Plain RAG,Agentic Loop
Control flow,Fixed: search → prompt → LLM,Dynamic: LLM decides what to do next
Number of searches,Always exactly one,As many as the model needs
Error recovery,None — bad search = bad answer,The LLM can retry with different keywords
Typo handling,Fails silently,"LLM can fix typos and re-search (e.g., ""Olama"" → ""Ollama"")"
LLM calls,One,Multiple (costs add up!)
Complexity,"Simple, predictable","More flexible, but harder to control costs"



Agent called search 6 times.

In [34]:
#result.all_messages